# DeepSeek-V4-Flash-Vision-Exp — DGX experiment ledger

**Current gate: MEASUREMENTS + PUBLICATION COMPLETE — PENDING MERGE + NODE RELEASE.** TP=2 service measured end to end: uncached prefill 1,789 tok/s client (engine peak 36,114.9 tok/s), decode c=1 36.9 tok/s / c=6 aggregate 112.7 tok/s, TTFT 0.239 s; text + vision smoke PASS; ClipProxy wired and live-verified (sentinel + vision 200). Limitation: 380K-token chat prompt stalls server-side pre-engine; largest verified prefill is 29,501 tokens.

Branch: codex/0.1.1-preflight | Publication surface: current draft PR


In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
LEDGER_PATH = ROOT / 'results' / 'ledger-state.json'
ledger = json.loads(LEDGER_PATH.read_text())

def phase(name):
    return ledger[name]

def require_gpu_clearance():
    preflight = phase('storage_preflight')
    auth = preflight['auth_preflight']
    ready = (
        preflight['go_no_go'] == 'GO'
        and not preflight['owner_safety_hold']
        and preflight['node_a']['pinned_snapshot_state'] == 'verified-complete'
        and preflight['node_b']['pinned_snapshot_state'] == 'verified-complete'
        and auth['node_a']['auth'] is True
        and auth['node_b']['auth'] is True
        and auth['download_context']['auth'] is True
    )
    if not ready:
        raise RuntimeError('NO-GO: owner safety/resource and per-node integrity gates are not all clear')
    return True

ledger['updated_utc']

## Live status and attempt table

This is the top operator checkpoint. Update and push it before and after every material gate or command expected to exceed 30 minutes.

In [ ]:
display(phase('top_status'))

columns = ['attempt', 'phase', 'single_change', 'blocker', 'expected_signal', 'stop_condition', 'result', 'evidence_path']
header = '| ' + ' | '.join(columns) + ' |'
separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
rows = [
    '| ' + ' | '.join(str(item.get(column, '')).replace('|', '\\|') for column in columns) + ' |'
    for item in phase('attempts')
]
display(Markdown('\n'.join([header, separator, *rows])))


## 1. Identity

Pinned model/runtime identities, proposed topology, quantization, protocol, and expected resource envelope. Planned TP=2 values are explicitly untested until a future cleared run.

In [ ]:
phase('identity')

## 2. Storage + preflight

Both node-local checkpoints now pass integrity, and the boolean Hub authentication preflight passes inside the exact detached-container context on both nodes. No credential value or lookup detail is stored.

In [ ]:
phase('storage_preflight')

### Authenticated-download preflight

Run the tracked preflight inside the exact non-interactive or container context that performs Hub requests. Its complete public receipt is one boolean field; credential material and lookup details are never printed or stored. A progressing resumable download remains untouched.


In [ ]:
auth_gate = phase('storage_preflight')['auth_preflight']
assert set(auth_gate['receipt_schema']) == {'auth'}
display({'node_a': auth_gate['node_a'], 'node_b': auth_gate['node_b'], 'download_context': auth_gate['download_context']})
# Exact-context command: python3 tools/hf_auth_preflight.py


## 3. Load gate

The historical single-node result is retained as measured capacity evidence. Its withdrawn cross-cluster source is not an approved recipe and must not be reused. The two-node load gate is untested.

In [ ]:
phase('load_gate')

### GPU execution guard

This cell intentionally fails while any safety, ownership, or per-node checkpoint-integrity gate is incomplete. Future download/load cells must remain below this guard and idempotent.

In [ ]:
require_gpu_clearance()
# No GPU/download command is committed while DGX_SAFETY_HOLD is active.

## 4. Functional gates

Text and vision outputs remain untested because no two-node service has reached ready state.

In [ ]:
phase('functional_gates')

## 5. Measurements

The required uncached prefill, fixed-concurrency decode, TTFT, aggregate/per-stream throughput, and reliability cells remain gated and have no invented values.

In [ ]:
phase('measurements')

## 6. Publication

Publication remains incomplete until a valid measured service provides text+vision output, prefill/decode/TTFT evidence, a best recipe, a live ClipProxy request with final usage, exact-head review, and merge.

In [ ]:
phase('publication')